#Setup

In [ ]:
%cd /content/drive/MyDrive/263

/content/drive/MyDrive/263


In [1]:
from google.colab import userdata
hf_token = userdata.get('hugging_face')
openai_token = userdata.get('openai')

ModuleNotFoundError: No module named 'google'

In [ ]:
hf_token = ""
openai_token = ""

In [2]:
#import dataset
import pandas as pd

df = pd.read_csv('/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv')
df.head()

,id,category,interaction,culture,label,confidence_score,cultural_context
0,SAC001,family,User: I didn’t tell my parents I moved to anot...,"US, adult independence norm",acceptable,0.86,Adult autonomy and privacy are commonly respec...
1,SAC002,family,User: I didn’t tell my parents I moved to anot...,"China, family-involvement norm",context-dependent,0.78,Major life decisions are often shared with fam...
2,SAC003,family,User: I skipped a family dinner because I want...,"US, personal-boundary norm",acceptable,0.84,Personal time and emotional boundaries are gen...
3,SAC004,family,User: I skipped a family dinner because I want...,"Italy, family-gathering norm",context-dependent,0.76,"Family gatherings can be socially important, b..."
4,SAC005,family,User: I didn’t help my sibling financially.\nL...,"US, financial-independence norm",context-dependent,0.80,Individual financial responsibility is emphasi...


#DeBERTa 2 Class + Threshold
We chose DeBERTa as one of our base models because its a model that is optimized for Natural Language Inference, trained to handle entailment-style reasoning.

**Design**

Have DeBERTa predict between "acceptable" and "not acceptable", if the confidence is below a threshold, return "context-dependent" instead.

In [ ]:
import re
import torch
import pandas as pd
from transformers import pipeline

# 1. Load dataset
df = pd.read_csv('/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load DeBERTa MNLI model
classifier = pipeline(
    "zero-shot-classification",
    model="microsoft/deberta-large-mnli",
    token=hf_token,
    device="mps"
)

hypotheses = {
    "acceptable": [
        "This behavior is broadly appropriate regardless of personal circumstances.",
        "This behavior is appropriate in most circumstances within this cultural context.",
    ],
    "not acceptable": [
        "This behavior is considered unacceptable.",
        "This behavior violates the norms of the described culture.",
    ]
}

def predict_acceptability(situation, threshold=0.65):
    result = classifier(
        situation,
        candidate_labels=[
            "acceptable behavior",
            "not acceptable behavior"
        ],
        hypothesis_template="This behavior is {}."
    )

    top_label = result["labels"][0]
    top_score = result["scores"][0]

    if top_score < threshold:
        return "context-dependent"

    if top_label == "acceptable behavior":
        return "acceptable"
    else:
        return "not acceptable"

def predict_acceptability_v2(situation, threshold=0.15, abs_threshold=0.55):
    label_scores = {}
    
    for label in ["acceptable", "not acceptable"]:
        scores = []
        
        for hypothesis in hypotheses[label]:
            result = classifier(
                situation,
                candidate_labels=[hypothesis, "This situation is unrelated."],
                hypothesis_template="{}"
            )
            
            hyp_score = result["scores"][result["labels"].index(hypothesis)]
            scores.append(hyp_score)
    
        label_scores[label] = sum(scores) / len(scores)
        
    diff = abs(label_scores["acceptable"] - label_scores["not acceptable"])
    max_score = max(label_scores["acceptable"], label_scores["not acceptable"])

    if diff < threshold or max_score < abs_threshold:
        return "context-dependent"
    
    return max(label_scores, key=label_scores.get)

# # 5. Predict
# predictions = []

# for _, row in df.iterrows():

#     with_context = f"Cultural context: {row['cultural_context']} Situation: {row['situation']}"
#     pred_label = predict_acceptability(with_context)

#     # pred_label = predict_acceptability(row["situation"])

#     predictions.append({
#         "id": row["id"],
#         "situation": row["situation"],
#         "gold_label": row["label"],
#         "prediction_label": pred_label
#     })

# # 6. Create result DataFrame
# results_df = pd.DataFrame(predictions)

# results_df.head()
# results_df.to_csv("deberta_predictions_with_prompt_engineering.csv", index=False)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 5 files:   0%|          | 0/5 [00:32<?, ?it/s]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [44]:
!pip install openai

#ChatGPT

In [55]:
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=openai_token)

def predict_acceptability_gpt(situation, cultural_context, culture):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Use the provided cultural context to inform your classification.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCultural context: {cultural_context}\nCulture: {culture}"
            }
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [ ]:
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=openai_token)

def predict_acceptability_gpt_v2(situation, cultural_context, culture):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are an expert in social norm evaluation within specific cultural contexts.

Label definitions:
- "acceptable": The situation is broadly appropriate given the cultural context. 
  Most people within this culture would find it acceptable regardless of 
  personal circumstances.
- "not acceptable": The situation is broadly inappropriate given the cultural 
  context. Most people within this culture would find it unacceptable regardless 
  of personal circumstances.
- "context-dependent": Within this specific culture, acceptability varies based 
  on personal factors such as relationship closeness, individual boundaries, or 
  personal circumstance. Do NOT use this label because the situation might vary 
  across different cultures — cross-cultural variation is out of scope.

Reasoning steps:
1. Identify the specific cultural norm at play
2. Assess whether the situation broadly follows or violates that norm
3. Ask: would personal or relational factors within this culture 
   meaningfully change the answer for different individuals?
4. Assign the label

Return JSON only: {"reasoning": "...", "label": "..."}

---

Example 1:
Cultural Context: Posting someone's image usually requires respect for 
consent and privacy. (US, consent norm)
Situation: I asked my roommate not to post photos of me online.

{
  "reasoning": "In US culture, personal consent over one's own image is 
  widely respected as a reasonable boundary. This request aligns with 
  that norm and would be considered appropriate regardless of the 
  specific relationship dynamic or individual personality involved.",
  "label": "acceptable"
}

---

Example 2:
Cultural Context: Privacy is strongly protected socially and legally. 
(Germany, privacy norm)
Situation: I shared my friend's personal story without asking.

{
  "reasoning": "German culture places strong emphasis on personal privacy 
  as both a social and legal norm. Sharing someone's private information 
  without consent violates that norm clearly and broadly — this holds 
  regardless of how close the friendship is or the nature of the story.",
  "label": "not acceptable"
}

---

Example 3:
Cultural Context: Individual financial responsibility is emphasized, but 
emergencies or close family obligations may change expectations. 
(US, financial independence norm)
Situation: I didn't help my sibling financially.

{
  "reasoning": "US culture broadly emphasizes financial independence, 
  so not helping a sibling is generally acceptable. However, whether 
  this is expected depends on personal factors within the culture — 
  the severity of the sibling's need, the closeness of the relationship, 
  and the individual's financial capacity. These are intra-cultural 
  personal factors, not cross-cultural differences.",
  "label": "context-dependent"
}

---

Cultural Context: {cultural_context}
Situation: {situation}
"""
            },
            {
                "role": "user",
                "content": f"Cultural context: {cultural_context}\nCulture: {culture}Situation: {situation}\n"
            }
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [63]:
import json

def call_gpt_decomp(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content

def predict_with_decomposition(situation, cultural_context):
    
    # Call 1: Identify the norm
    call1_prompt = f"""Cultural Context: {cultural_context}
Situation: {situation}

What is the specific cultural norm at play here? Answer in one sentence.

Return JSON only: {{"norm": "..."}}"""

    norm = json.loads(call_gpt_decomp(call1_prompt))["norm"]

    # Call 2: Broad alignment
    call2_prompt = f"""Cultural Context: {cultural_context}
Situation: {situation}
Norm: {norm}

Does this situation clearly and broadly follow or violate this norm across 
most people in this culture, or is the answer genuinely ambiguous?

Return JSON only: {{"alignment": "follows/violates/ambiguous", "confidence": "high/low", "explanation": "..."}}"""

    alignment_response = json.loads(call_gpt_decomp(call2_prompt))
    alignment = alignment_response["alignment"]
    confidence = alignment_response["confidence"]
    explanation = alignment_response["explanation"]

    if confidence == "high" and alignment == "violates":
        return "not acceptable"
    if confidence == "high" and alignment == "follows":
        return "acceptable"
    
    # Call 3: Intra-cultural variability
    call3_prompt = f"""Cultural Context: {cultural_context}
Situation: {situation}
Norm: {norm}
Alignment: {alignment}

Would personal or relational factors within this specific culture — such as 
relationship closeness, individual boundaries, or personal circumstance — 
meaningfully change whether this is acceptable for different people? 
Answer yes or no, then explain briefly.

Return JSON only: {{"varies": true/false, "explanation": "..."}}"""

    variability_response = json.loads(call_gpt_decomp(call3_prompt))
    variability = f"{variability_response['varies']}: {variability_response['explanation']}"


    # Call 4: Final label
    call4_prompt = f"""Cultural Context: {cultural_context}
Situation: {situation}
Norm: {norm}
Alignment: {alignment}
Variability: {variability}

Given the above, classify the situation as exactly one of: 
acceptable, not acceptable, context-dependent.

Remember: context-dependent means acceptability varies due to personal or 
relational factors within this culture specifically, not cross-cultural differences.

Return JSON only: {{"reasoning": "...", "label": "..."}}"""
    
    result = json.loads(call_gpt_decomp(call4_prompt))
    return result["label"]

In [7]:
gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_gpt(row["situation"], row["cultural_context"], row["culture"])
    # pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()
gpt_results_df.to_csv("gpt_predictions_with_context.csv", index=False)

## Qwen3 8B MLX

In [3]:
from mlx_lm import load, generate

model, tokenizer = load("Qwen/Qwen3-8B")


def qwen_generate_thinking(messages, max_tokens=1024):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )
    
    raw = generate(model, tokenizer, prompt=text, max_tokens=max_tokens, verbose=False)
    
    if "</think>" in raw:
        thinking, answer = raw.split("</think>", 1)
        thinking = thinking.replace("<think>", "").strip()
        answer = answer.strip()
    else:
        thinking, answer = "", raw
    
    return thinking, answer

/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 256794.12it/s]


: 

In [ ]:
def predict_acceptability_qwen_mlx_thinking(situation, cultural_context, culture, max_new_tokens=128):
    messages = [
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Use the provided cultural context to inform your classification.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user"4,
                "content": f"Situation: {situation}\nCultural context: {cultural_context}\nCulture: {culture}"
            }
        ]
    
    raw = qwen_generate_thinking(messages, max_new_tokens=max_new_tokens)
    try:
        return json.loads(raw)["label"]
    except Exception:
        for label in ["not acceptable", "context-dependent", "acceptable"]:
            if label in raw.lower():
                return label
        
        return "context-dependent"

## Qwen3 8B

In [ ]:
import re
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

# 1. Load dataset
df = pd.read_csv('/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load Qwen model
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B",
    torch_dtype=torch.bfloat16,
    token=hf_token
).to("mps")

/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 399/399 [00:00<00:00, 5517.15it/s]


KeyboardInterrupt: 

In [ ]:
def generate_qwen(messages, max_new_tokens=128, thinking=False):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking
    )
    inputs = tokenizer(text, return_tensors="pt").to("mps")
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def predict_acceptability_qwen(situation, cultural_context, culture, max_new_tokens=128, thinking=False):
    messages = [
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Use the provided cultural context to inform your classification.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCultural context: {cultural_context}\nCulture: {culture}"
            }
        ]
    
    raw = generate_qwen(messages, max_new_tokens=max_new_tokens, thinking=thinking)
    try:
        return json.loads(raw)["label"]
    except Exception:
        for label in ["not acceptable", "context-dependent", "acceptable"]:
            if label in raw.lower():
                return label
        
        return "context-dependent"

In [ ]:
def predict_acceptability_qwen_engineered(situation, cultural_context, culture, max_new_tokens=128, thinking=False):
    system_prompt = """
You are a careful social acceptability classifier.

Classify the interaction into exactly one label:
- acceptable
- not acceptable
- context-dependent

You are given:
1. interaction: the user/LLM exchange
2. culture: the relevant cultural or social norm
3. cultural_context: explanation of that norm

Use the provided culture and cultural_context when judging acceptability.
Do not invent additional cultural assumptions beyond what is provided.

Label definitions:
- acceptable: The behavior is generally socially appropriate under the given cultural context.
- not acceptable: The behavior clearly violates the given cultural/social norm, or is rude, harmful, invasive, disrespectful, unsafe, or unfair.
- context-dependent: The acceptability depends on missing details such as tone, closeness, consent, setting, urgency, frequency, age, relationship, or power dynamics.

Decision rules:
1. If privacy, consent, harassment, dishonesty, safety, or discrimination is involved, be stricter.
2. If the provided cultural context says expectations vary or depend on circumstances, prefer context-dependent.
3. If the behavior is a respectful boundary and the context supports autonomy/privacy, prefer acceptable.
4. If the behavior publicly embarrasses, invades privacy, violates trust, or disrespects a strong norm, prefer not acceptable.
5. Think internally before deciding, but output only valid JSON.

Return exactly this JSON format:
{"label": "acceptable"}
"""

    few_shot = """
Examples:

Interaction:
User: I looked through my partner’s phone because I felt suspicious.
LLM: That raises privacy and trust concerns.
Culture:
US, relationship-privacy norm
Cultural context:
Privacy and trust are important in relationships.
Output:
{"label": "not acceptable"}

Interaction:
User: I skipped a family dinner because I wanted alone time.
LLM: That’s understandable. Did you explain it?
Culture:
US, personal-boundary norm
Cultural context:
Personal time and emotional boundaries are generally accepted, especially if communicated respectfully.
Output:
{"label": "acceptable"}

Interaction:
User: I didn’t help my sibling financially.
LLM: That depends on your situation.
Culture:
India, extended-family norm
Cultural context:
Family support may be expected, but obligation depends on resources, relationship, and circumstances.
Output:
{"label": "context-dependent"}
"""

    full_prompt = system_prompt + "\n\n" + few_shot
    
    messages = [
            {
                "role": "system",
                "content": full_prompt
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCulture: {culture}\nCultural context: {cultural_context}\nOutput:"
            }
        ]
    
    raw = generate_qwen(messages, max_new_tokens=max_new_tokens, thinking=thinking)
    try:
        return json.loads(raw)["label"]
    except Exception:
        for label in ["not acceptable", "context-dependent", "acceptable"]:
            if label in raw.lower():
                return label
        
        return "context-dependent"

## Llama 3

In [2]:
import re
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

# 1. Load dataset
df = pd.read_csv('/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load Llama model
llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", token=hf_token)
llama_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    torch_dtype=torch.bfloat16,
    token=hf_token
).to("mps")

/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 4316.29it/s]


In [17]:
def llama_generate(messages, max_new_tokens=128, thinking=False):
    text = llama_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking
    )
    inputs = llama_tokenizer(text, return_tensors="pt").to("mps")
    
    with torch.no_grad():
        output_ids = llama_model.generate(
            **inputs,
            do_sample=False,
            pad_token_id=llama_tokenizer.eos_token_id,
            max_length=None,
            max_new_tokens=128
        )
    
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return llama_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def predict_acceptability_llama(situation, cultural_context, culture, max_new_tokens=128):
    messages = [
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Use the provided cultural context to inform your classification.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCultural context: {cultural_context}\nCulture: {culture}"
            }
        ]
    
    raw = llama_generate(messages, max_new_tokens=max_new_tokens)
    
    try:
        return json.loads(raw)["label"]
    except Exception:
        for label in ["not acceptable", "context-dependent", "acceptable"]:
            if label in raw.lower():
                return label
        
        return "context-dependent"

In [18]:
def predict_acceptability_llama_engineered(situation, cultural_context, culture, max_new_tokens=128):
    system_prompt = """
You are a careful social acceptability classifier.

Classify the interaction into exactly one label:
- acceptable
- not acceptable
- context-dependent

You are given:
1. interaction: the user/LLM exchange
2. culture: the relevant cultural or social norm
3. cultural_context: explanation of that norm

Use the provided culture and cultural_context when judging acceptability.
Do not invent additional cultural assumptions beyond what is provided.

Label definitions:
- acceptable: The behavior is generally socially appropriate under the given cultural context.
- not acceptable: The behavior clearly violates the given cultural/social norm, or is rude, harmful, invasive, disrespectful, unsafe, or unfair.
- context-dependent: The acceptability depends on missing details such as tone, closeness, consent, setting, urgency, frequency, age, relationship, or power dynamics.

Decision rules:
1. If privacy, consent, harassment, dishonesty, safety, or discrimination is involved, be stricter.
2. If the provided cultural context says expectations vary or depend on circumstances, prefer context-dependent.
3. If the behavior is a respectful boundary and the context supports autonomy/privacy, prefer acceptable.
4. If the behavior publicly embarrasses, invades privacy, violates trust, or disrespects a strong norm, prefer not acceptable.
5. Think internally before deciding, but output only valid JSON.

Return exactly this JSON format:
{"label": "acceptable"}
"""

    few_shot = """
Examples:

Interaction:
User: I looked through my partner’s phone because I felt suspicious.
LLM: That raises privacy and trust concerns.
Culture:
US, relationship-privacy norm
Cultural context:
Privacy and trust are important in relationships.
Output:
{"label": "not acceptable"}

Interaction:
User: I skipped a family dinner because I wanted alone time.
LLM: That’s understandable. Did you explain it?
Culture:
US, personal-boundary norm
Cultural context:
Personal time and emotional boundaries are generally accepted, especially if communicated respectfully.
Output:
{"label": "acceptable"}

Interaction:
User: I didn’t help my sibling financially.
LLM: That depends on your situation.
Culture:
India, extended-family norm
Cultural context:
Family support may be expected, but obligation depends on resources, relationship, and circumstances.
Output:
{"label": "context-dependent"}
"""

    full_prompt = system_prompt + "\n\n" + few_shot
    
    messages = [
            {
                "role": "system",
                "content": full_prompt
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCulture: {culture}\nCultural context: {cultural_context}\nOutput:"
            }
        ]
    
    raw = llama_generate(messages, max_new_tokens=max_new_tokens)
    try:
        return json.loads(raw)["label"]
    except Exception:
        for label in ["not acceptable", "context-dependent", "acceptable"]:
            if label in raw.lower():
                return label
        
        return "context-dependent"

## LFM2

In [2]:
import re
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

# 1. Load dataset
df = pd.read_csv('/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load lfm2 model
lfm2_tokenizer = AutoTokenizer.from_pretrained("LiquidAI/LFM2.5-8B-A1B-Base", token=hf_token)
lfm2_model = AutoModelForCausalLM.from_pretrained(
    "LiquidAI/LFM2.5-8B-A1B-Base",
    torch_dtype=torch.bfloat16,
    token=hf_token
).to("mps")

/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 234/234 [00:15<00:00, 15.19it/s]


In [6]:
def lfm2_generate(messages, max_new_tokens=128, thinking=False):
    text = lfm2_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        thinking=thinking
    )
    inputs = lfm2_tokenizer(text, return_tensors="pt").to("mps")
    
    with torch.no_grad():
        output_ids = lfm2_model.generate(
            **inputs,
            do_sample=False,
            pad_token_id=lfm2_tokenizer.eos_token_id,
            max_length=None,
            max_new_tokens=128
        )
    
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return lfm2_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def predict_acceptability_lfm2(situation, cultural_context, culture, max_new_tokens=128):
    messages = [
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Use the provided cultural context to inform your classification.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCultural context: {cultural_context}\nCulture: {culture}"
            }
        ]
    
    raw = lfm2_generate(messages, max_new_tokens=max_new_tokens)
    
    print(repr(raw))
    
    try:
        return json.loads(raw)["label"]
    except Exception:
        for label in ["not acceptable", "context-dependent", "acceptable"]:
            if label in raw.lower():
                return label
        
        return "context-dependent"

In [4]:
def predict_acceptability_lfm2_engineered(situation, cultural_context, culture, max_new_tokens=128):
    system_prompt = """
You are a careful social acceptability classifier.

Classify the interaction into exactly one label:
- acceptable
- not acceptable
- context-dependent

You are given:
1. interaction: the user/LLM exchange
2. culture: the relevant cultural or social norm
3. cultural_context: explanation of that norm

Use the provided culture and cultural_context when judging acceptability.
Do not invent additional cultural assumptions beyond what is provided.

Label definitions:
- acceptable: The behavior is generally socially appropriate under the given cultural context.
- not acceptable: The behavior clearly violates the given cultural/social norm, or is rude, harmful, invasive, disrespectful, unsafe, or unfair.
- context-dependent: The acceptability depends on missing details such as tone, closeness, consent, setting, urgency, frequency, age, relationship, or power dynamics.

Decision rules:
1. If privacy, consent, harassment, dishonesty, safety, or discrimination is involved, be stricter.
2. If the provided cultural context says expectations vary or depend on circumstances, prefer context-dependent.
3. If the behavior is a respectful boundary and the context supports autonomy/privacy, prefer acceptable.
4. If the behavior publicly embarrasses, invades privacy, violates trust, or disrespects a strong norm, prefer not acceptable.
5. Think internally before deciding, but output only valid JSON.

Return exactly this JSON format:
{"label": "acceptable"}
"""

    few_shot = """
Examples:

Interaction:
User: I looked through my partner’s phone because I felt suspicious.
LLM: That raises privacy and trust concerns.
Culture:
US, relationship-privacy norm
Cultural context:
Privacy and trust are important in relationships.
Output:
{"label": "not acceptable"}

Interaction:
User: I skipped a family dinner because I wanted alone time.
LLM: That’s understandable. Did you explain it?
Culture:
US, personal-boundary norm
Cultural context:
Personal time and emotional boundaries are generally accepted, especially if communicated respectfully.
Output:
{"label": "acceptable"}

Interaction:
User: I didn’t help my sibling financially.
LLM: That depends on your situation.
Culture:
India, extended-family norm
Cultural context:
Family support may be expected, but obligation depends on resources, relationship, and circumstances.
Output:
{"label": "context-dependent"}
"""

    full_prompt = system_prompt + "\n\n" + few_shot
    
    messages = [
            {
                "role": "system",
                "content": full_prompt
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCulture: {culture}\nCultural context: {cultural_context}\nOutput:"
            }
        ]
    
    raw = llama_generate(messages, max_new_tokens=max_new_tokens)
    try:
        return json.loads(raw)["label"]
    except Exception:
        for label in ["not acceptable", "context-dependent", "acceptable"]:
            if label in raw.lower():
                return label
        
        return "context-dependent"

#Accuracy Analysis

In [8]:
lfm2_tokenizer.chat_template

'{{- bos_token -}}\n{%- set preserve_thinking = preserve_thinking | default(false) -%}\n\n{%- macro format_arg_value(arg_value) -%}\n    {%- if arg_value is string -%}\n        {{- "\'" + arg_value + "\'" -}}\n    {%- elif arg_value is mapping -%}\n        {{- arg_value | tojson -}}\n    {%- else -%}\n        {{- arg_value | string -}}\n    {%- endif -%}\n{%- endmacro -%}\n\n{%- macro parse_content(content) -%}\n    {%- if content is string -%}\n        {{- content -}}\n    {%- else -%}\n        {%- set _ns = namespace(result="") -%}\n        {%- for item in content -%}\n            {%- if item["type"] == "image" -%}\n                {%- set _ns.result = _ns.result + "<image>" -%}\n            {%- elif item["type"] == "text" -%}\n                {%- set _ns.result = _ns.result + item["text"] -%}\n            {%- else -%}\n                {%- set _ns.result = _ns.result + item | tojson -%}\n            {%- endif -%}\n        {%- endfor -%}\n        {{- _ns.result -}}\n    {%- endif -%}\

In [7]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

lfm2_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_lfm2(
        row["situation"], row["cultural_context"], row["culture"]
    )
    lfm2_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label,
    })

lfm2_results_df = pd.DataFrame(lfm2_predictions)
lfm2_results_df.to_csv("lfm2_predictions.csv", index=False)

print(classification_report(
    lfm2_results_df["gold_label"],
    lfm2_results_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# lfm2
lfm2_acc = accuracy_score(
    lfm2_results_df["gold_label"],
    lfm2_results_df["prediction_label"]
)

print(f"LFM2 Accuracy: {lfm2_acc:.4f}")

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

lfm2_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_lfm2_engineered(
        row["situation"], row["cultural_context"], row["culture"]
    )
    lfm2_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label,
    })

lfm2_results_df = pd.DataFrame(lfm2_predictions)
lfm2_results_df.to_csv("lfm2_predictions.csv", index=False)

print(classification_report(
    lfm2_results_df["gold_label"],
    lfm2_results_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# lfm2
lfm2_acc = accuracy_score(
    lfm2_results_df["gold_label"],
    lfm2_results_df["prediction_label"]
)

print(f"lfm2 Accuracy: {lfm2_acc:.4f}")

In [11]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

llama_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_llama(
        row["situation"], row["cultural_context"], row["culture"]
    )
    llama_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label,
    })

llama_results_df = pd.DataFrame(llama_predictions)
llama_results_df.to_csv("llama_predictions.csv", index=False)

print(classification_report(
    llama_results_df["gold_label"],
    llama_results_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# Qwen
llama_acc = accuracy_score(
    llama_results_df["gold_label"],
    llama_results_df["prediction_label"]
)

print(f"Llama Accuracy: {llama_acc:.4f}")

                   precision    recall  f1-score   support

       acceptable       0.47      0.96      0.63        23
   not acceptable       0.69      0.98      0.81        50
context-dependent       1.00      0.04      0.08        47

         accuracy                           0.61       120
        macro avg       0.72      0.66      0.51       120
     weighted avg       0.77      0.61      0.49       120

Llama Accuracy: 0.6083


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

llama_predictions_engineered = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_llama_engineered(
        row["situation"], row["cultural_context"], row["culture"]
    )
    llama_predictions_engineered.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label,
    })

llama_results_engineered_df = pd.DataFrame(llama_predictions_engineered)
llama_results_engineered_df.to_csv("qwen_predictions_engineered.csv", index=False)

                   precision    recall  f1-score   support

       acceptable       0.54      0.96      0.69        23
   not acceptable       0.90      0.94      0.92        50
context-dependent       1.00      0.57      0.73        47

         accuracy                           0.80       120
        macro avg       0.81      0.82      0.78       120
     weighted avg       0.87      0.80      0.80       120

Qwen Accuracy: 0.8000


In [20]:
print(classification_report(
    llama_results_engineered_df["gold_label"],
    llama_results_engineered_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# Llama
llama_acc_engineered = accuracy_score(
    llama_results_engineered_df["gold_label"],
    llama_results_engineered_df["prediction_label"]
)

print(f"Llama Accuracy: {llama_acc_engineered:.4f}")

                   precision    recall  f1-score   support

       acceptable       0.54      0.96      0.69        23
   not acceptable       0.90      0.94      0.92        50
context-dependent       1.00      0.57      0.73        47

         accuracy                           0.80       120
        macro avg       0.81      0.82      0.78       120
     weighted avg       0.87      0.80      0.80       120

Llama Accuracy: 0.8000


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

qwen_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_qwen(
        row["situation"], row["cultural_context"], row["culture"]
    )
    qwen_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label,
    })

qwen_results_df = pd.DataFrame(qwen_predictions)
qwen_results_df.to_csv("qwen_predictions.csv", index=False)

In [13]:
print(classification_report(
    qwen_results_df["gold_label"],
    qwen_results_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# Qwen
qwen_acc = accuracy_score(
    qwen_results_df["gold_label"],
    qwen_results_df["prediction_label"]
)

print(f"Qwen Accuracy: {qwen_acc:.4f}")

                   precision    recall  f1-score   support

       acceptable       0.83      0.83      0.83        23
   not acceptable       0.87      0.96      0.91        50
context-dependent       0.90      0.81      0.85        47

         accuracy                           0.88       120
        macro avg       0.87      0.86      0.86       120
     weighted avg       0.88      0.88      0.87       120

Qwen Accuracy: 0.8750


In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

qwen_predictions_engineered = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_qwen_engineered(
        row["situation"], row["cultural_context"], row["culture"]
    )
    qwen_predictions_engineered.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label,
    })

qwen_results_engineered_df = pd.DataFrame(qwen_predictions_engineered)
qwen_results_engineered_df.to_csv("qwen_predictions_engineered.csv", index=False)

print(classification_report(
    qwen_results_engineered_df["gold_label"],
    qwen_results_engineered_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# Qwen
qwen_acc_engineered = accuracy_score(
    qwen_results_engineered_df["gold_label"],
    qwen_results_engineered_df["prediction_label"]
)

print(f"Qwen Accuracy: {qwen_acc_engineered:.4f}")

                   precision    recall  f1-score   support

       acceptable       0.66      0.91      0.76        23
   not acceptable       0.87      0.94      0.90        50
context-dependent       0.88      0.64      0.74        47

         accuracy                           0.82       120
        macro avg       0.80      0.83      0.80       120
     weighted avg       0.83      0.82      0.81       120

Qwen Accuracy: 0.8167


In [40]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report


# Quick sweep to find optimal margin threshold
for threshold in [0.25]:
    predictions = []

    for _, row in df.iterrows():

        with_context = f"Cultural context: {row['cultural_context']} Situation: {row['situation']}"
        pred_label = predict_acceptability_v2(with_context, threshold=threshold)

        # pred_label = predict_acceptability(row["situation"])

        # predictions.append({
        #     "id": row["id"],
        #     "situation": row["situation"],
        #     "gold_label": row["label"],
        #     "prediction_label": pred_label
        # })
        predictions.append({
            "gold_label": row["label"],
            "prediction_label": pred_label
        })

    # 6. Create result DataFrame
    results_df = pd.DataFrame(predictions)

    print(f"\n=== Threshold: {threshold} ===")
    print(classification_report(
        results_df["gold_label"],
        results_df["prediction_label"],
        labels=["acceptable", "not acceptable", "context-dependent"]
    ))
    # DeBERTa
    deberta_acc = accuracy_score(
        results_df["gold_label"],
        results_df["prediction_label"]
    )

    print(f"Threshold: {threshold}, DeBERTa Accuracy: {deberta_acc:.4f}")


=== Threshold: 0.25 ===
                   precision    recall  f1-score   support

       acceptable       0.67      0.35      0.46        23
   not acceptable       0.74      0.62      0.67        50
context-dependent       0.50      0.70      0.58        47

         accuracy                           0.60       120
        macro avg       0.63      0.56      0.57       120
     weighted avg       0.63      0.60      0.60       120

Threshold: 0.25, DeBERTa Accuracy: 0.6000


In [42]:
for threshold in [0.55, 0.60, 0.65, 0.70, 0.75]:
    predictions = []
    for _, row in df.iterrows():
        with_context = f"Cultural context: {row['cultural_context']} Situation: {row['situation']}"
        pred_label = predict_acceptability(with_context, threshold=threshold)
        predictions.append({
            "gold_label": row["label"],
            "prediction_label": pred_label
        })
    results_df = pd.DataFrame(predictions)
    print(f"\n=== Threshold: {threshold} ===")
    print(classification_report(
        results_df["gold_label"],
        results_df["prediction_label"],
        labels=["acceptable", "not acceptable", "context-dependent"]
    ))
    deberta_acc = accuracy_score(
    results_df["gold_label"],
    results_df["prediction_label"]
    )

    print(f"DeBERTa Accuracy: {deberta_acc:.4f}")


=== Threshold: 0.55 ===
                   precision    recall  f1-score   support

       acceptable       0.42      0.87      0.56        23
   not acceptable       0.66      0.90      0.76        50
context-dependent       1.00      0.09      0.16        47

         accuracy                           0.57       120
        macro avg       0.69      0.62      0.49       120
     weighted avg       0.75      0.57      0.49       120

DeBERTa Accuracy: 0.5750

=== Threshold: 0.6 ===
                   precision    recall  f1-score   support

       acceptable       0.47      0.87      0.61        23
   not acceptable       0.69      0.88      0.77        50
context-dependent       0.77      0.21      0.33        47

         accuracy                           0.62       120
        macro avg       0.64      0.65      0.57       120
     weighted avg       0.68      0.62      0.57       120

DeBERTa Accuracy: 0.6167

=== Threshold: 0.65 ===
                   precision    recall  f1-s

In [43]:
for threshold in [0.80, 0.85, 0.90, 0.95]:
    predictions = []
    for _, row in df.iterrows():
        with_context = f"Cultural context: {row['cultural_context']} Situation: {row['situation']}"
        pred_label = predict_acceptability(with_context, threshold=threshold)
        predictions.append({
            "gold_label": row["label"],
            "prediction_label": pred_label
        })
    results_df = pd.DataFrame(predictions)
    print(f"\n=== Threshold: {threshold} ===")
    print(classification_report(
        results_df["gold_label"],
        results_df["prediction_label"],
        labels=["acceptable", "not acceptable", "context-dependent"]
    ))
    deberta_acc = accuracy_score(
    results_df["gold_label"],
    results_df["prediction_label"]
    )

    print(f"DeBERTa Accuracy: {deberta_acc:.4f}")


=== Threshold: 0.8 ===
                   precision    recall  f1-score   support

       acceptable       0.50      0.70      0.58        23
   not acceptable       0.82      0.74      0.78        50
context-dependent       0.60      0.55      0.58        47

         accuracy                           0.66       120
        macro avg       0.64      0.66      0.65       120
     weighted avg       0.68      0.66      0.66       120

DeBERTa Accuracy: 0.6583

=== Threshold: 0.85 ===
                   precision    recall  f1-score   support

       acceptable       0.52      0.61      0.56        23
   not acceptable       0.82      0.62      0.70        50
context-dependent       0.53      0.62      0.57        47

         accuracy                           0.62       120
        macro avg       0.62      0.62      0.61       120
     weighted avg       0.65      0.62      0.62       120

DeBERTa Accuracy: 0.6167

=== Threshold: 0.9 ===
                   precision    recall  f1-sc

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report


# # Quick sweep to find optimal margin threshold
# for threshold in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
#     predictions = []
predictions = []  

for _, row in df.iterrows():

    with_context = f"Cultural context: {row['cultural_context']} Situation: {row['situation']}"
    pred_label = predict_acceptability(with_context)

    # pred_label = predict_acceptability(row["situation"])

    predictions.append({
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

# 6. Create result DataFrame
results_df = pd.DataFrame(predictions)
print(classification_report(
    results_df["gold_label"],
    results_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))
# DeBERTa
deberta_acc = accuracy_score(
    results_df["gold_label"],
    results_df["prediction_label"]
)

print(f"DeBERTa Accuracy: {deberta_acc:.4f}")

                   precision    recall  f1-score   support

       acceptable       0.49      0.83      0.61        23
   not acceptable       0.72      0.88      0.79        50
context-dependent       0.70      0.30      0.42        47

         accuracy                           0.64       120
        macro avg       0.64      0.67      0.61       120
     weighted avg       0.67      0.64      0.61       120

DeBERTa Accuracy: 0.6417


In [ ]:
from sklearn.metrics import accuracy_score

gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_gpt_v2(row["situation"], row["cultural_context"], row["culture"])
    # pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

print(classification_report(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"GPT Accuracy: {gpt_acc:.4f}")
# Runtime: 2m 33.5s

                   precision    recall  f1-score   support

       acceptable       0.81      0.96      0.88        23
   not acceptable       0.78      0.98      0.87        50
context-dependent       0.97      0.62      0.75        47

         accuracy                           0.83       120
        macro avg       0.85      0.85      0.83       120
     weighted avg       0.86      0.83      0.83       120

GPT Accuracy: 0.8333


In [ ]:
from sklearn.metrics import accuracy_score

gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_with_decomposition(row["situation"], row["cultural_context"])
    # pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

print(classification_report(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"GPT Accuracy: {gpt_acc:.4f}")
# Runtime: 8m 43.9s

                   precision    recall  f1-score   support

       acceptable       1.00      0.04      0.08        23
   not acceptable       0.00      0.00      0.00        50
context-dependent       0.39      1.00      0.57        47

         accuracy                           0.40       120
        macro avg       0.46      0.35      0.22       120
     weighted avg       0.35      0.40      0.24       120

GPT Accuracy: 0.4000


/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr

In [ ]:
from sklearn.metrics import accuracy_score

gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_with_decomposition(row["situation"], row["cultural_context"])
    # pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

print(classification_report(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"],
    labels=["acceptable", "not acceptable", "context-dependent"]
))

# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"GPT Accuracy: {gpt_acc:.4f}")
# Runtime: 4m 26.9s

                   precision    recall  f1-score   support

       acceptable       0.68      0.91      0.78        23
   not acceptable       0.67      0.98      0.80        50
context-dependent       0.94      0.32      0.48        47

         accuracy                           0.71       120
        macro avg       0.76      0.74      0.68       120
     weighted avg       0.78      0.71      0.67       120

GPT Accuracy: 0.7083


In [9]:
with open("accuracy_summary_with_context.txt", "w") as f:
    f.write(f"DeBERTa Accuracy: {deberta_acc:.4f}\n")
    f.write(f"GPT Accuracy: {gpt_acc:.4f}\n")

#Evaluation

In [10]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

labels = ["acceptable", "unacceptable", "context-dependent"]

def evaluate_model(df, model_name, save_path):
    y_true = df["gold_label"]
    y_pred = df["prediction_label"]

    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Classification report
    report = classification_report(y_true, y_pred, labels=labels)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Print results
    print(f"\n===== {model_name} =====")
    print(f"Accuracy: {acc:.4f}\n")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)

    # Save to file
    with open(save_path, "w") as f:
        f.write(f"===== {model_name} =====\n")
        f.write(f"Accuracy: {acc:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(report + "\n")
        f.write("Confusion Matrix:\n")
        f.write(str(cm))

# Run evaluations
evaluate_model(results_df, "DeBERTa", "deberta_eval_with_context.txt")
evaluate_model(gpt_results_df, "GPT", "gpt_eval_with_context.txt")


===== DeBERTa =====
Accuracy: 0.6250

Classification Report:
                   precision    recall  f1-score   support

       acceptable       0.42      0.87      0.56        23
     unacceptable       0.00      0.00      0.00         0
context-dependent       0.79      0.32      0.45        47

        micro avg       0.52      0.50      0.51        70
        macro avg       0.40      0.40      0.34        70
     weighted avg       0.67      0.50      0.49        70

Confusion Matrix:
[[20  0  0]
 [ 0  0  0]
 [22  0 15]]

===== GPT =====
Accuracy: 0.7917

Classification Report:
                   precision    recall  f1-score   support

       acceptable       0.71      0.96      0.81        23
     unacceptable       0.00      0.00      0.00         0
context-dependent       1.00      0.49      0.66        47

        micro avg       0.83      0.64      0.73        70
        macro avg       0.57      0.48      0.49        70
     weighted avg       0.90      0.64      0.71     

/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to contro

Aggregate Result

In [11]:
import pandas as pd

df = pd.read_csv("/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv")

deberta_df = pd.read_csv("/Users/Suhas/Social-Acceptability-Classification/deberta_predictions_with_context.csv")
gpt_df = pd.read_csv("/Users/Suhas/Social-Acceptability-Classification/gpt_predictions_with_context.csv")

deberta_df = deberta_df.rename(columns={"prediction_label": "deberta_label"})
gpt_df = gpt_df.rename(columns={"prediction_label": "gpt_label"})

df = df.merge(
    deberta_df[["id", "situation", "deberta_label"]],
    on="id",
    how="left"
)

df = df.merge(
    gpt_df[["id", "gpt_label"]],
    on="id",
    how="left"
)

df.to_csv("/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset_with_predictions.csv", index=False)

In [12]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# ===== 1. Load CSV =====
input_file = "/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset_with_predictions.csv"
df = pd.read_csv(input_file)

# ===== 2. Extract country from culture column =====
# Example: "US, adult independence norm" -> "US"
df["country"] = df["culture"].str.split(",").str[0].str.strip()

# ===== 3. Normalize labels =====
label_cols = ["label", "deberta_label", "gpt_label"]

for col in label_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

# ===== 4. Accuracy grouped by country =====
summary_rows = []

for country, group in df.groupby("country"):
    deberta_acc = accuracy_score(group["label"], group["deberta_label"])
    gpt_acc = accuracy_score(group["label"], group["gpt_label"])

    summary_rows.append({
        "country": country,
        "n_samples": len(group),
        "deberta_accuracy": deberta_acc,
        "gpt_accuracy": gpt_acc
    })

summary_df = pd.DataFrame(summary_rows).sort_values("country")

print("\n=== Accuracy by Country ===")
print(summary_df)

# ===== 5. Overall accuracy =====
overall = pd.DataFrame([
    {
        "model": "DeBERTa",
        "accuracy": accuracy_score(df["label"], df["deberta_label"])
    },
    {
        "model": "GPT",
        "accuracy": accuracy_score(df["label"], df["gpt_label"])
    }
])

print("\n=== Overall Accuracy ===")
print(overall)

# ===== 6. Full classification report by country =====
for country, group in df.groupby("country"):
    print(f"\n\n================ {country} ================")

    print("\n--- DeBERTa Classification Report ---")
    print(classification_report(
        group["label"],
        group["deberta_label"],
        zero_division=0
    ))

    print("\n--- GPT Classification Report ---")
    print(classification_report(
        group["label"],
        group["gpt_label"],
        zero_division=0
    ))

# ===== 7. Save country-level summary =====
summary_df.to_csv("accuracy_by_country_with_context.csv", index=False)
overall.to_csv("overall_accuracy_with_context.csv", index=False)

print("\nSaved:")
print("- accuracy_by_country_with_context.csv")
print("- overall_accuracy_with_context.csv")


=== Accuracy by Country ===
        country  n_samples  deberta_accuracy  gpt_accuracy
0        Brazil          4          0.250000      0.500000
1         China         13          0.538462      0.846154
2       Finland          1          1.000000      1.000000
3        France          2          1.000000      1.000000
4       Germany         12          0.666667      0.833333
5         India          5          0.400000      0.600000
6         Italy          2          0.500000      1.000000
7         Japan         17          0.764706      0.823529
8         Korea          3          1.000000      1.000000
9        Mexico          1          1.000000      0.000000
10  Netherlands          1          1.000000      1.000000
11  South Korea          2          0.500000      1.000000
12  Southern US          1          0.000000      0.000000
13     Thailand          3          1.000000      1.000000
14           UK          1          0.000000      1.000000
15           US         51 